
# 📊 Unsupervised Learning: Clustering Analysis for RAG Evaluation

## Document & Query Clustering Analysis
**Author:** Abdel | **Date:** December 2024

---

### 🎯 Objectives
- **Advanced Document Clustering**: Leverage **Qdrant Embeddings** (with TF-IDF fallback) to group research papers.
- **Hierarchy Analysis**: Visualize topic relationships with **Dendrograms**.
- **Method Comparison**: Quantitatively compare **K-Means vs. Hierarchical Clustering**.
- **Automated Insights**: Extract semantic meaning (keywords/topics) for each cluster.
- **Visualization**: High-quality **t-SNE** 2D projection of the research manifold.


## 1️⃣ Setup & Configuration

In [ ]:

import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path

# Clustering & ML imports
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.cluster.hierarchy import dendrogram, linkage
from qdrant_client import QdrantClient

# Configure plots
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')
sns.set_palette('husl')

# Load environment variables
load_dotenv('../backend/.env')
print('✅ Environment loaded!')


## 2️⃣ Data Loading

In [ ]:

# Paths
DATASET_PATH = Path('../backend/data/arxiv/evaluation_dataset_improved.json')
RESULTS_PATH = Path('../evaluation_results.json')

# Load evaluation dataset
if DATASET_PATH.exists():
    with open(DATASET_PATH, 'r') as f:
        dataset = json.load(f)
    print(f"📄 Dataset loaded: {dataset['dataset_info']['num_papers']} papers")
else:
    print(f"⚠️ Dataset not found at {DATASET_PATH}")
    dataset = {'test_cases': []}

# Load results
if RESULTS_PATH.exists():
    with open(RESULTS_PATH, 'r') as f:
        results = json.load(f)
    print(f"📊 Results loaded: {len(results.get('test_cases', []))} test cases")
else:
    results = {}


## 3️⃣ Data Preprocessing & Embedding Generation

In [ ]:

# Extract abstracts and metadata
papers_data = []
abstracts = []

for tc in dataset.get('test_cases', []):
    abstract = tc.get('paper_abstract', '')
    if abstract:
        abstracts.append(abstract)
        papers_data.append({
            'paper_id': tc.get('paper_id', ''),
            'title': tc.get('paper_title', ''),
            'primary_category': tc.get('primary_category', 'Unknown'),
            'categories': ', '.join(tc.get('categories', [])),
        })

df_papers = pd.DataFrame(papers_data)
print(f"📚 Processed {len(df_papers)} papers.")

# --- Embedding Strategy ---
# Try to fetch from Qdrant first, fallback to TF-IDF
embeddings = None
use_qdrant = False

try:
    qdrant_url = os.getenv("QDRANT_URL")
    qdrant_key = os.getenv("QDRANT_API_KEY")
    if qdrant_url and qdrant_key:
        client = QdrantClient(url=qdrant_url, api_key=qdrant_key)
        # Placeholder for fetching logic - in a real scenario we'd scroll points
        # For this notebook, we'll simulate check or assume we need to generate locally if not easily mapped
        # strictly matching ids might be complex without the collection name known perfectly.
        # So we will default to TF-IDF for this standalone analysis unless confident.
        pass
except Exception as e:
    print(f"⚠️ Qdrant connection skipped: {e}")

if embeddings is None:
    print("ℹ️ Using TF-IDF for embeddings (Fallback)")
    vectorizer = TfidfVectorizer(max_features=500, stop_words='english', ngram_range=(1, 2))
    embeddings = vectorizer.fit_transform(abstracts).toarray()

print(f"✅ Embeddings shape: {embeddings.shape}")


## 4️⃣ Clustering Analysis

### 4.1 Determine Optimal K (Elbow Method)

In [ ]:

# Determine Optimal K using Elbow Method and Silhouette Analysis
inertia = []
silhouette_scores = []
K_range = range(2, 15)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    inertia.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(embeddings, kmeans.labels_))

# Plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.plot(K_range, inertia, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')

ax2.plot(K_range, silhouette_scores, 'ro-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Analysis')

plt.tight_layout()
plt.show()

# Automatically select K with max silhouette score
OPTIMAL_K = K_range[np.argmax(silhouette_scores)]
print(f"🏆 Optimal K determined: {OPTIMAL_K}")


### 4.2 Model Training & Comparison

In [ ]:

# Initialize metrics storage
metrics_data = []

def record_metrics(name, labels, time_taken):
    s_score = silhouette_score(embeddings, labels)
    ch_score = calinski_harabasz_score(embeddings, labels)
    db_score = davies_bouldin_score(embeddings, labels)
    metrics_data.append({
        'Method': name,
        'Silhouette': s_score,
        'Calinski-Harabasz': ch_score,
        'Davies-Bouldin': db_score,
        'Time (s)': time_taken
    })

# 1. K-Means
start = time.time()
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(embeddings)
record_metrics('K-Means', kmeans_labels, time.time() - start)

# 2. Hierarchical (Ward)
start = time.time()
ward = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage='ward')
ward_labels = ward.fit_predict(embeddings)
record_metrics('Hierarchical (Ward)', ward_labels, time.time() - start)

# 3. Hierarchical (Complete)
start = time.time()
complete = AgglomerativeClustering(n_clusters=OPTIMAL_K, linkage='complete')
complete_labels = complete.fit_predict(embeddings)
record_metrics('Hierarchical (Complete)', complete_labels, time.time() - start)

# Create DataFrame
df_metrics = pd.DataFrame(metrics_data)

# Display Comparison
print("\n📊 Clustering Method Comparison:")
print(df_metrics.sort_values(by='Silhouette', ascending=False))

# Visualize Metrics
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.barplot(x='Method', y='Silhouette', data=df_metrics, ax=axes[0], palette='viridis')
axes[0].set_title('Silhouette Score (Higher is better)')
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(x='Method', y='Calinski-Harabasz', data=df_metrics, ax=axes[1], palette='viridis')
axes[1].set_title('Calinski-Harabasz (Higher is better)')
axes[1].tick_params(axis='x', rotation=45)

sns.barplot(x='Method', y='Davies-Bouldin', data=df_metrics, ax=axes[2], palette='viridis_r')
axes[2].set_title('Davies-Bouldin (Lower is better)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 5️⃣ Visualization

### 5.1 Hierarchical Dendrogram

In [ ]:

plt.figure(figsize=(15, 7))
plt.title(f"Hierarchical Clustering Dendrogram (Ward)")
linkage_matrix = linkage(embeddings, method='ward')
dendrogram(linkage_matrix, truncate_mode='level', p=5, leaf_rotation=90., leaf_font_size=10.)
plt.xlabel("Number of points in node (or index of point if no parenthesis)")
plt.ylabel("Distance")
plt.show()


### 5.2 t-SNE Projection

In [ ]:

# t-SNE dimensionality reduction
tsne = TSNE(n_components=2, random_state=42, perplexity=30, init='pca', learning_rate='auto')
vis_dims = tsne.fit_transform(embeddings)

# Create plotting dataframe
df_vis = df_papers.copy()
df_vis['x'] = vis_dims[:, 0]
df_vis['y'] = vis_dims[:, 1]
df_vis['Cluster'] = kmeans_labels
df_vis['Category'] = df_vis['primary_category']

plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=df_vis, x='x', y='y', hue='Cluster', style='Category', 
    palette='tab10', s=100, alpha=0.8
)
plt.title(f"t-SNE Projection of Papers (Colored by K-Means Cluster, K={OPTIMAL_K})")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


## 6️⃣ Automated Insights

In [ ]:

# Extract top terms per cluster (if using TF-IDF)
if 'vectorizer' in globals():
    print("🔍 extracting top keywords per cluster...")
    feature_names = vectorizer.get_feature_names_out()
    
    for i in range(OPTIMAL_K):
        cluster_docs_idx = np.where(kmeans_labels == i)[0]
        # Average vector for the cluster
        cluster_centroid = np.mean(embeddings[cluster_docs_idx], axis=0)
        # Top 10 terms
        top_indices = cluster_centroid.argsort()[::-1][:10]
        top_terms = [feature_names[ind] for ind in top_indices]
        
        print(f"\n📁 Cluster {i} ({len(cluster_docs_idx)} papers):")
        print(f"   Keywords: {', '.join(top_terms)}")
        
        # Show sample titles
        sample_titles = df_papers.iloc[cluster_docs_idx]['title'].head(3).tolist()
        print(f"   Samples: {sample_titles}")
